In [ ]:
import digitalhub as dh

project = dh.get_or_create_project("demo-guardrails")

# 1. Deploy LLM

We will use KubeAI runtime to deploy a simple model to be protected by guardrails.

In [ ]:
llm_function = project.new_function("llm",
                                    kind="kubeai-text",
                                    model_name="gemma3",
                                    url="ollama://gemma3:latest",
                                    engine="OLlama")

# 2. Create Guardrail Service with Guardrails AI 

We will use ``guardrails`` runtime function.

## 2.1 Creating a function

In [ ]:
from pathlib import Path
Path("src").mkdir(exist_ok=True)

In [ ]:
%%writefile "src/guardrail_service.py"

import nuclio_sdk
import os
import json
from guardrails import Guard, OnFailAction

def init_context(context: nuclio_sdk.Context):
    context.logger.info("Initializing guardrails...")
    from guardrails.hub import ToxicLanguage
    guard = Guard().use(
        ToxicLanguage, threshold=0.5, validation_method="sentence", on_fail="exception"
    )
    setattr(context, "guard", guard)

def handler_serve(context: nuclio_sdk.Context, event: nuclio_sdk.Event):
    if isinstance(event.body, bytes):
        body = json.loads(event.body)
    else:
        body = event.body
        
    prompt = body['prompt'] if 'prompt' in body else None

    if prompt:
        try:
            guard.validate(prompt) 
        except Exception as e:
            return context.Response(body="Toxic language used",
                            headers={},
                            content_type='text/plain',
                            status_code=400)

    return event.body

In [ ]:
func = project.new_function(name="toxic-guardrail",
                            kind="guardrail",
                            python_version="PYTHON3_10",
                            code_src="src/guardrail_service.py",
                            handler="handler_serve",
                            init_function="init_context",
                            processing_mode="preprocessor",
                            requirements=["guardrails-ai==0.5.0", "transformers==4.42.0"]
                           )

## 2.2 Building a function image

The guardrails-ai library is based on predefined or custom guardrail validators. Predefined validators may be obtained from the Guardrails AI hub, downloaded and deployed. In this scenario, we will use predefined guardrails that should be integrated in the service. To make it efficient, the guardrails should be integrated in the underlying container image and we will build it for this function.

First, we need an API KEY from Guardrails AI to access the hub. We will add to the project as a secret.

In [ ]:
secret = project.new_secret(name="GUARDRAILS_API_KEY",
                            secret_value="value")

To build the image for the function we will need to add some instructions to use Git, to authenticate to the hub, and to install the specific validator (toxic_language).

In [ ]:
build_run = func.run(action="build", 
                     secrets=["GUARDRAILS_API_KEY"],
                     instructions=[
                         "/opt/nuclio/uv/uv pip install --system  typer==0.9.0 click==8.1.7 guardrails-ai==0.5.0",
                         "apt-get update && apt-get install -y git",
                         "--mount=type=secret,id=GUARDRAILS_API_KEY,env=GUARDRAILS_API_KEY guardrails configure --enable-metrics --enable-remote-inferencing --token $GUARDRAILS_API_KEY",
                         "guardrails hub install hub://guardrails/toxic_language"
                     ]
                    )

## 2.3 Running the functions

We need to deploy the guard and protect LLM with it.

In [ ]:
guardrail_run = func.run(action="serve")

Run the LLM with guardrails extension. The extension will add the model to the AI gateway that will intercepts the calls and interact with the guardrails.

In [ ]:
llm_run = llm_function.run(action="serve", extensions=[{
    "kind": "envoygw",
    "name": "gw",
    "spec": {
        "guardrails": [guardrail_run.refresh().status.service['url']]
    }
}])

Test the function is up and running: see models exposed

In [ ]:
import requests

BASE_URL = llm_run.refresh().status.service["url"]

res = requests.get(f"{BASE_URL}/models")
res.json()

## 2.4 Test the functions

Test the function is up and running: make a completion call

In [ ]:
import requests

BASE_URL = llm_run.refresh().status.service['url']
MODEL = llm_run.status.openai["model"]
data = {
    "model": MODEL,
    "prompt": "My landlord is an asshole!"
  }

res = requests.post(f"{BASE_URL}/completions", json=data)
res.json()

Test the guardrails: call the LLM through protected gateway. The AI Gateway exposes the OpenAI models under /v1 path.

In [ ]:
import requests

BASE_URL = f"{llm_run.refresh().status.gatewayInfo['gatewayEndpoint']}/v1"
MODEL = llm_run.status.openai["model"]
data = {
    "model": MODEL,
    "prompt": "My landlord is an asshole!"
  }

res = requests.post(f"http://{BASE_URL}/completions", json=data)
res.text